# Patch events (D9)

Measures the patch rule in `sql/patch_events.sql` (views `news_patch_flags`, `patch_events`).

what share of the 7,381 posts the rule flags
which rule has the most false positives
what share of real patches the rule misses

In [6]:
import os, sys, pandas as pd, psycopg
from dotenv import load_dotenv
sys.path.insert(0, "..")
from collector.parse import is_patch
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)
def q(sql, **params): return pd.read_sql(sql, conn, params=params)

## Feeds

D9's title rule assumes developer posts only; this should show only `steam_community_announcements`.

In [7]:
q("SELECT feedname, count(*) FROM news GROUP BY 1")

C:\Users\dillo\AppData\Local\Temp\ipykernel_18800\3630970414.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  def q(sql, **params): return pd.read_sql(sql, conn, params=params)


,feedname,count
0,steam_community_announcements,7482


## Counts per rule

The rules overlap, so tag + keyword + version is more than flagged.

In [8]:
q("""SELECT count(*) FILTER (WHERE rule_tag OR rule_keyword OR rule_version) AS flagged, count(*) AS posts,
            count(*) FILTER (WHERE rule_tag) AS tag, count(*) FILTER (WHERE rule_keyword) AS keyword,
            count(*) FILTER (WHERE rule_version) AS version
     FROM news_patch_flags""")

C:\Users\dillo\AppData\Local\Temp\ipykernel_18800\3630970414.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  def q(sql, **params): return pd.read_sql(sql, conn, params=params)


,flagged,posts,tag,keyword,version
0,3026,7482,1757,2550,1251


## Parity: SQL vs Python

The D9 rule lives in two places: `is_patch` in `collector/parse.py` and `news_patch_flags` in `sql/patch_events.sql`. They must agree on every post; any disagreement shows which version is wrong.

In [9]:
posts = q("SELECT gid, title, tags FROM news")
posts["python"] = [is_patch({"title": r.title, "tags": r.tags}) for r in posts.itertuples()]
flags = q("SELECT gid, rule_tag, rule_keyword, rule_version FROM news_patch_flags")
both = posts.merge(flags, on="gid", validate="one_to_one")
both["sql"] = both.rule_tag | both.rule_keyword | both.rule_version
bad = both[both.python != both.sql]
print(f"{len(both):,} posts; {len(bad)} where is_patch and the SQL rules disagree")
bad[["gid", "title", "tags", "python", "sql"]]

7,482 posts; 0 where is_patch and the SQL rules disagree


C:\Users\dillo\AppData\Local\Temp\ipykernel_18800\3630970414.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  def q(sql, **params): return pd.read_sql(sql, conn, params=params)
C:\Users\dillo\AppData\Local\Temp\ipykernel_18800\3630970414.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  def q(sql, **params): return pd.read_sql(sql, conn, params=params)


,gid,title,tags,python,sql


## Hand-check sample

Four groups, each defined by the first rule that fires, so every post is in exactly one: `tag` (the tag rule's precision),
`keyword_no_tag` (what the keyword rule adds), `version_only` (what the version rule adds), `none` (the misses).
10 posts per group, at most one per game, ordered by `md5(gid)`: the same sample on every run.

Labeling is blind: `patch_posts_to_label.csv` has no group column and is shuffled. Don't open `patch_sample_key.csv` until every row is labeled.
The cell won't overwrite existing files, so rerunning it can't wipe your labels.

In [10]:
LABELS = "../data/labels"
STRATA = ["tag", "keyword_no_tag", "version_only", "none"]

sample = q("""
WITH s AS (
    SELECT f.gid, f.appid, f.title, n.url, left(n.contents, 500) AS excerpt,
           CASE WHEN f.rule_tag     THEN 'tag'
                WHEN f.rule_keyword THEN 'keyword_no_tag'
                WHEN f.rule_version THEN 'version_only'
                ELSE 'none' END AS stratum
    FROM news_patch_flags f JOIN news n USING (gid)
), one_per_game AS (
    SELECT *, row_number() OVER (PARTITION BY stratum, appid ORDER BY md5(gid)) AS nth
    FROM s
), picked AS (
    SELECT *, row_number() OVER (PARTITION BY stratum ORDER BY md5(gid)) AS k
    FROM one_per_game WHERE nth = 1
)
SELECT gid, stratum, title, url, excerpt FROM picked WHERE k <= 10""")
assert (sample.stratum.value_counts() == 10).all() and len(sample) == 40

key_path, label_path = f"{LABELS}/patch_sample_key.csv", f"{LABELS}/patch_posts_to_label.csv"
if os.path.exists(key_path) or os.path.exists(label_path):
    print("label files already exist; not overwriting")
else:
    os.makedirs(LABELS, exist_ok=True)
    sample[["gid", "stratum"]].to_csv(key_path, index=False)
    to_label = sample[["gid", "title", "url", "excerpt"]].assign(is_patch_true="", note="")
    to_label.sample(frac=1, random_state=0).to_csv(label_path, index=False, encoding="utf-8-sig")  # BOM: Excel reads UTF-8
    print(f"wrote {len(to_label)} posts to label")

label files already exist; not overwriting


C:\Users\dillo\AppData\Local\Temp\ipykernel_18800\3630970414.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  def q(sql, **params): return pd.read_sql(sql, conn, params=params)


Each group's total size, for weighting the groups' results into overall numbers.

In [11]:
sizes = q("""SELECT CASE WHEN rule_tag     THEN 'tag'
                WHEN rule_keyword THEN 'keyword_no_tag'
                WHEN rule_version THEN 'version_only'
                ELSE 'none' END AS stratum, count(*) AS posts
             FROM news_patch_flags GROUP BY 1""").set_index("stratum").posts.reindex(STRATA)
sizes

C:\Users\dillo\AppData\Local\Temp\ipykernel_18800\3630970414.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  def q(sql, **params): return pd.read_sql(sql, conn, params=params)


stratum
tag               1757
keyword_no_tag    1163
version_only       106
none              4456
Name: posts, dtype: int64

In [14]:
LABEL_PATH = "../data/labels/patch_posts_to_label.csv"
lab = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", dtype={"gid": str})   # dtype=str keeps all 19 digits
for i, r in lab.iterrows():
    print(f"[{i}] {r.title}\n     {r.url}\n     {str(r.excerpt)[:500]!r}\n")

[0] Patch 1.52.a
     https://steamstore-a.akamaihd.net/news/externalpost/steam_community_announcements/1819386365126270
     '[h2]Happy New Year Scouts! [/h2][p]We have a small patch for you today adding more maps to the map rotation and removing the holiday stuff. If you liked making snow balls, don’t worry! The snow piles will remain on Alpine.\n[/p][h3]The Steam Awards [/h3][p]PEAK won the better with friends category!!! Thank you all so much, it’s the first steam award for both Landfall and Aggro Crab and it means a lot to us that it was voted on by all of you.\n[/p][h3]Patch notes [/h3][list][*][p]Removed the holida'

[1] Minor Update - Main branch
     https://steamstore-a.akamaihd.net/news/externalpost/steam_community_announcements/1811772772601830
     "[p]Hello Terraformers!\n\nWe've just uploaded a small patch on the main branch. \nYou need to restart Steam to get it. \n[/p][list][*][p]This fix a potential security breach found in the engine that we use (Unity). It should ha

In [15]:
answers = {
    0: (1, ""),
    1: (0, "roadmap, nothing shipped yet"),
    2: (1, "hotfix"),
    # ... through 39
}

In [16]:
assert set(answers) == set(lab.index), f"missing rows: {sorted(set(lab.index) - set(answers))}"
assert all(v[0] in (0, 1) for v in answers.values())
lab["is_patch_true"] = [answers[i][0] for i in lab.index]
lab["note"] = [answers[i][1] for i in lab.index]
lab.to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
print(lab.is_patch_true.value_counts())

AssertionError: missing rows: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]

## Results (after labeling)

Precision per flagged group and the patch rate in `none`, each with a 95% Wilson interval. Overall precision weights each
flagged group by its size (equal samples from unequal groups, so a plain average would be wrong). Estimated misses =
patch rate in `none` × size of `none`. With only 10 unflagged posts the recall estimate is rough; the range below uses the ends
of `none`'s interval, holding precision fixed.

In [13]:
import math

def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, centre - half), min(1.0, centre + half)

# Hand-written statistics: checked against known 95% intervals
assert tuple(round(v, 3) for v in wilson(8, 10)) == (0.490, 0.943)
assert tuple(round(v, 3) for v in wilson(0, 10)) == (0.000, 0.278)

labels = pd.read_csv(f"{LABELS}/patch_posts_to_label.csv", encoding="utf-8-sig", dtype={"gid": str})
key = pd.read_csv(f"{LABELS}/patch_sample_key.csv", encoding="utf-8-sig", dtype={"gid": str})
# gids are 18-19 digits; Excel saves them as 5.12E+18 with the last digits zeroed, so rows stop joining (or join wrongly)
assert set(labels.gid) == set(key.gid) and len(labels) == len(key) == 40, "labeled gids don't match the key"
assert labels.is_patch_true.isin([0, 1]).all(), "label every row 0 or 1 first"
lab = labels.merge(key, on="gid", validate="one_to_one")

res = lab.groupby("stratum").is_patch_true.agg(patches="sum", n="count").reindex(STRATA)
res["rate"] = res.patches / res.n
res[["lo", "hi"]] = [wilson(k, n) for k, n in zip(res.patches, res.n)]
res["posts"] = sizes
display(res.round(2))

flagged = res.loc[["tag", "keyword_no_tag", "version_only"]]
true_flagged = (flagged.rate * flagged.posts).sum()
print(f"overall precision {true_flagged / flagged.posts.sum():.0%} (weighted by group size)")
recall = lambda rate_none: true_flagged / (true_flagged + rate_none * res.posts["none"])
miss = res.loc["none"]
print(f"estimated misses {miss.rate * miss.posts:,.0f} of {int(miss.posts):,} unflagged posts")
print(f"recall {recall(miss.rate):.0%}, rough range {recall(miss.hi):.0%} to {recall(miss.lo):.0%} (only {miss.n:.0f} unflagged posts checked)")
lab[lab.note.notna()][["stratum", "is_patch_true", "title", "note"]]

AssertionError: label every row 0 or 1 first